# Querying

Ask questions about your video collection using the Responses API. This notebook covers basic queries, instructions, response parsing, and common query patterns.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "your_store_id")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## When You Need This

You have a knowledge store with indexed videos and want to query it using natural language.

## Helper: Parse Response

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""

## Your First Query

Send a natural language question to the Responses API with your knowledge store as a tool.

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": "What are the main themes across these videos and images?",
        }
    ],
)

print(parse_response(response))

## Using Instructions

Specialize Jockey's behavior for your domain by providing an `instructions` field. This acts as a system prompt that shapes how Jockey interprets and responds to queries.

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    instructions=(
        "You are a sports analyst. Focus on player performance, "
        "tactics, and key moments."
    ),
    input=[
        {
            "type": "message",
            "role": "user",
            "content": "Summarize the key plays from this game",
        }
    ],
)

print(parse_response(response))

## Reading the Full Response

The response contains metadata beyond the message content — session IDs for multi-turn conversations, status, and token usage.

In [ ]:
# Inspect the full response structure
print(f"ID: {response.id}")
print(f"Session: {response.session_id}")
print(f"Status: {response.status}")
print(f"Tokens: {response.usage}")

# Iterate through all output blocks
for output in response.output:
    if output.type == "message":
        for content in output.content:
            print(f"\nMessage content:\n{content.text}")

---

## Query Patterns

The Responses API handles a wide range of query types through the same endpoint. Below are the most common patterns.

### Pattern 1: Collection Overview

Get a high-level corpus digest — themes, subjects, patterns, and key statistics.

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "Give me a comprehensive overview of these videos and images. "
                "Include main themes, recurring subjects, content types, "
                "and any notable patterns."
            ),
        }
    ],
)

print(parse_response(response))

### Pattern 2: Search and Discovery

Find specific moments, topics, or content with natural language. Use a JSON schema in the `text` field to get structured results.

In [ ]:
SEARCH_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "item_reference": {"type": "string"},
                    "timestamp": {"type": "string"},
                    "description": {"type": "string"},
                    "relevance": {"type": "string"},
                },
            },
        },
        "total_results": {"type": "integer"},
        "query_interpretation": {"type": "string"},
    },
}

response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": "Find all moments where someone is presenting to an audience",
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="search_results", schema_=SEARCH_SCHEMA)
    ),
)

search_results = json.loads(parse_response(response))
print(json.dumps(search_results, indent=2))

### Search Query Examples

| Query | What It Finds |
|-------|---------------|
| "someone laughing" | Moments with laughter |
| "product being held up to camera" | Product showcase moments |
| "outdoor scenes with water" | Nature/water visuals |
| "heated discussion" | Tense conversational moments |
| "text on screen" | Moments with overlaid text or titles |

**Tips:**
- Narrow by context with instructions: "Only search the first 2 minutes of each video"
- Rank results: "Find and rank the top 5 most visually striking moments"
- Refine in multi-turn: search, then follow up with "Show me more like the third result"

### Pattern 3: Entity Extraction

List all entities (people, places, objects, brands, concepts) across your videos with structured output.

In [ ]:
ENTITY_SCHEMA = {
    "type": "object",
    "properties": {
        "entities": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "type": {"type": "string"},
                    "frequency": {"type": "string"},
                    "appears_in": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                },
            },
        },
        "entity_count": {"type": "integer"},
    },
}

response = client.responses.create(
    knowledge_store_id=STORE_ID,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "List every distinct entity across all videos and images — people, places, "
                "objects, brands, and concepts. Include how frequently each appears."
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="entity_list", schema_=ENTITY_SCHEMA)
    ),
)

entities = json.loads(parse_response(response))
print(f"Found {entities.get('entity_count', 'N/A')} entities:")
print(json.dumps(entities, indent=2))

### Entity Extraction Tips

- **Filter by type:** "List only the people who appear in these videos"
- **Cross-video tracking:** "Which entities appear in more than one video?"
- **With relationships:** "List entities and how they relate to each other"

## Common Pitfalls

- **Knowledge store must have ready items.** If no items are `ready`, the query has nothing to reason over.
- **Knowledge store ID required.** Every request must include a `knowledge_store_id` field.
- **Input format matters.** Each message needs `type`, `role`, and `content` fields.

## Next Steps

- [Multi-Turn Sessions](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response/multi-turn-sessions) — continue conversations across multiple turns
- [Streaming](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response/streaming) — get responses in real-time
- [Structured Output](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response/structured-output) — get typed JSON back
- [Authentication](authentication.ipynb) — API key setup and security
- [Building Knowledge Stores](building_knowledge_stores.ipynb) — create and populate stores

**API Reference:** [POST /responses](https://twelvelabs-preview-7f6af7ac-b5df-4358-a7dc-8573e931a808.docs.buildwithfern.com/api-reference/responses/create-response)